# Step 2 — Anchor-Fixed Semantic Walk with Basket Hopping

Run the stochastic walk and save the **raw, unfiltered** trajectories. This is the first of a two-stage split: this notebook only decides *where* each walk goes. Separate filter notebooks (`02a_filter_*`) turn these raw paths into training sentences. Splitting them this way guarantees every threshold method operates on the exact same walk trajectories — `theta` only ever decides what gets logged into a sentence, never where the walk goes next.

The walk hops between baskets using real transaction data, but every candidate is scored against a single **fixed anchor** (the product the walk started from) — never against the most recently visited product. This prevents the semantic drift that a chain of locally-plausible steps would otherwise cause.

Step 3 (Skip-gram training), full-dataset scale-up, and the Step 4 LLM judge are explicitly out of scope here — as is any thresholding, which lives in the filter notebooks.

**Inputs**
- `outputs/embeddings.npy`, `outputs/product_id_to_index.json`, `outputs/products_text.csv` (Step 1)
- `data/order_products__prior.csv`

**Outputs** (to `outputs/`)
- `walk_raw_paths.jsonl` — one raw walk per line (`anchor_id`, `raw_path`, `raw_sims`, `hops_taken`), every walk regardless of length
- `walk_config.json` — the exact walk-mechanics settings + resolved sample sizes + timestamp behind this run
- `product_basket_freq.json` — per-product sampled-basket counts + total basket count, for downstream popularity checks
- `shared_sample.json` — the canonical sampled order IDs + starting products, so downstream consumers (incl. the cluster runner `run_walk_cluster.py`) reuse the exact sample instead of re-drawing from seeds

## 1. Imports and paths

Paths are resolved relative to the project root so the notebook works regardless of where Jupyter is launched from.

In [1]:
import json
import datetime
import sys
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
STEP1_DIR = OUT_DIR / "step1"

# Make project-root modules (walk_lib) importable regardless of the launch dir.
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from walk_lib import apply_score_transform, softmax, sim_to_anchor, run_walk

print("Data dir:   ", DATA_DIR)
print("Outputs dir:", OUT_DIR)

Data dir:    /Users/linnhtet/Developer/bachelor-thesis/data
Outputs dir: /Users/linnhtet/Developer/bachelor-thesis/outputs


## 2. Parameters

All tunable knobs live here so they are easy to adjust between runs.

**`starting_products`** is defined by the two knobs below and resolved in §6: set `STARTING_PRODUCTS_EXPLICIT` to a list of `product_id`s to walk from those exactly, or leave it `None` to randomly sample `N_STARTING_PRODUCTS` walkable products. Total walk count is `len(starting_products) * r`. The six stress-test anchors (used by the filter notebooks' validation) are always added regardless.

Note there is **no** `theta` or `min_sentence_length` here: this notebook does no thresholding. Those knobs belong to the filter notebooks that consume the raw paths this one produces.

In [2]:
# ---- Walk mechanics ----
L = 15                     # max hops per walk
r = 20                     # walks generated per starting product
tau = 0.15                 # softmax temperature for winner sampling
max_consecutive_empty = 3  # terminate after this many empty-candidate hops in a row

# score transform applied to similarities *before* the softmax (spec §6.4).
# One of: "identity" (default), "squared", "min_max_in_basket".
score_transform = "identity"

# ---- Reproducibility (two independent seeds, spec §4) ----
random_seed = 42           # walk stochasticity (winner sampling, basket choice)
sample_seed = 7            # order sampling — kept separate from walk seed

# ---- Data scope (spec §4) ----
N_ORDERS_SAMPLE = 1_000_000  # complete orders to sample

# ---- Starting products (spec §7 note) ----
STARTING_PRODUCTS_EXPLICIT = None   # e.g. [4210, 6347]; None -> random sample
N_STARTING_PRODUCTS = 20000         # sample size when EXPLICIT is None

# ---- Stress-test anchors (spec §10) — always walked, needed by the filter notebooks ----
# Same product_ids as 01_semantic_vectorization.ipynb.
STRESS_TEST_IDS = {
    4210:  "Whole Milk",
    6347:  "Unsweetened Almond Milk",
    651:   "Organic Salted Butter",
    16268: "Vegan Butter",
    6104:  "Whole Milk Plain Yogurt",
    920:   "Coconut Yogurt",
}

## 3. Load Step 1 artifacts

Load the embeddings, the `product_id -> row` map, and the product metadata. We L2-normalise every embedding row **once** here, so cosine similarity against the anchor later reduces to a plain dot product (spec §6.3).

In [3]:
embeddings = np.load(STEP1_DIR / "embeddings.npy").astype(np.float32)
# Normalise to unit length -> cosine similarity becomes a dot product (spec §6.3).
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = embeddings / np.clip(norms, 1e-12, None)

with open(STEP1_DIR / "product_id_to_index.json") as f:
    pid_to_idx = {int(pid): row for pid, row in json.load(f).items()}

products_text = pd.read_csv(STEP1_DIR / "products_text.csv")
id_to_name = dict(zip(products_text["product_id"], products_text["product_name"]))
catalog_ids = set(pid_to_idx)


def name(pid):
    return id_to_name.get(int(pid), f"?{pid}")


def names(ids):
    return [name(p) for p in ids]


print(f"embeddings: {embeddings.shape} | catalog products: {len(catalog_ids):,}")

embeddings: (49688, 384) | catalog products: 49,688


## 4. Sample orders

`order_products__prior.csv` has 32.4M rows; per spec §4 we do **not** walk on the full file. We read it once (the one unavoidable full pass), sample a fixed number of complete orders with `sample_seed`, and keep each sampled basket's full contents (never truncate a basket). Then we log the diagnostics the spec asks for.

In [4]:
prior = pd.read_csv(
    DATA_DIR / "order_products__prior.csv",
    usecols=["order_id", "product_id"],
    dtype={"order_id": "int32", "product_id": "int32"},
)

all_order_ids = prior["order_id"].unique()
n_sample = min(N_ORDERS_SAMPLE, len(all_order_ids))
sample_rng = np.random.default_rng(sample_seed)
sampled_ids = sample_rng.choice(all_order_ids, size=n_sample, replace=False)
sampled_set = set(int(x) for x in sampled_ids)

sample = prior[prior["order_id"].isin(sampled_set)]

# ---- Diagnostics (spec §4) ----
basket_sizes = sample.groupby("order_id").size()
covered = set(int(p) for p in sample["product_id"].unique())
zero_basket = catalog_ids - covered

print(f"sampled orders           : {n_sample:,} of {len(all_order_ids):,}")
print(f"rows kept                : {len(sample):,}")
print(f"unique products covered  : {len(covered):,}")
print(f"basket size mean/median  : {basket_sizes.mean():.2f} / {basket_sizes.median():.0f}")
print(f"basket size min/max      : {basket_sizes.min()} / {basket_sizes.max()}")
print(f"catalog products with 0 baskets in sample: {len(zero_basket):,} "
      f"({100 * len(zero_basket) / len(catalog_ids):.1f}%) "
      f"-- cannot be walked from, noted not dropped")

sampled orders           : 1,000,000 of 3,214,874
rows kept                : 10,087,780
unique products covered  : 48,526
basket size mean/median  : 10.09 / 8
basket size min/max      : 1 / 137
catalog products with 0 baskets in sample: 1,162 (2.3%) -- cannot be walked from, noted not dropped


## 5. Build the basket graph

Two plain dictionaries (spec §5), built once from the sample and reused across every walk:

- `product_to_orders`: `product_id -> set(order_id)` — which baskets contain it
- `order_to_products`: `order_id -> [product_id, ...]` — a basket's full contents

In [5]:
product_to_orders = defaultdict(set)
order_to_products = defaultdict(list)

for oid, pid in zip(sample["order_id"].to_numpy(), sample["product_id"].to_numpy()):
    oid, pid = int(oid), int(pid)
    product_to_orders[pid].add(oid)
    order_to_products[oid].append(pid)

product_to_orders = dict(product_to_orders)
order_to_products = dict(order_to_products)

print(f"product_to_orders: {len(product_to_orders):,} products")
print(f"order_to_products: {len(order_to_products):,} baskets")

product_to_orders: 48,526 products
order_to_products: 1,000,000 baskets


## 6. Resolve `starting_products`

Turn the §2 knobs into the concrete list of anchors. We only sample from products that actually appear in the sample (a product with no baskets cannot be walked from). The stress-test anchors are always appended, and any starting product with zero baskets is reported (spec §4: note, don't silently drop).

In [6]:
walkable = set(product_to_orders)   # products with >= 1 basket in the sample

if STARTING_PRODUCTS_EXPLICIT is not None:
    starting_products = list(STARTING_PRODUCTS_EXPLICIT)
else:
    pool = np.array(sorted(walkable))
    start_rng = np.random.default_rng(sample_seed + 1)  # separate deterministic stream
    k = min(N_STARTING_PRODUCTS, len(pool))
    starting_products = [int(x) for x in start_rng.choice(pool, size=k, replace=False)]

# Always include the stress-test anchors (spec §10).
for pid in STRESS_TEST_IDS:
    if pid not in starting_products:
        starting_products.append(pid)

no_baskets = [pid for pid in starting_products if pid not in walkable]
print(f"starting_products: {len(starting_products)} "
      f"(incl. {len(STRESS_TEST_IDS)} stress-test anchors)")
if no_baskets:
    print(f"  WARNING: {len(no_baskets)} have no baskets in the sample and will be "
          f"skipped: {[name(p) for p in no_baskets]}")
else:
    print("  all starting products have at least one basket to walk from")

starting_products: 20005 (incl. 6 stress-test anchors)
  all starting products have at least one basket to walk from


In [7]:
# Persist the canonical shared sample so every downstream consumer (incl. the
# cluster runner run_walk_cluster.py) loads the exact same orders + anchors
# instead of re-drawing from the seeds. This decouples the project from numpy's
# RNG stream matching bit-for-bit across numpy versions or platforms.
shared_sample = {
    "sampled_order_ids": [int(x) for x in sampled_ids],   # exact orders drawn in §4
    "starting_products": [int(p) for p in starting_products],
    "sample_seed": sample_seed,
    "n_orders_sample": n_sample,
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
}
with open(OUT_DIR / "shared_sample.json", "w") as f:
    json.dump(shared_sample, f)
print(f"Saved shared_sample.json: {len(shared_sample['sampled_order_ids']):,} orders, "
      f"{len(shared_sample['starting_products']):,} starting products")

Saved shared_sample.json: 1,000,000 orders, 20,005 starting products


## 7. Scoring helpers

`score_transform` reshapes the similarity distribution *before* the softmax (spec §6.4). The **raw** cosine similarity (pre-transform) is what we record per hop as `raw_sims`, so the filter notebooks can threshold against the same Step 1 scale the walk actually saw.

These helpers now live in `walk_lib.py` (imported in §1) — the single source of truth shared with the cluster runner `run_walk_cluster.py`, so the two implementations can't drift apart.

## 8. The walk

One walk from anchor `A`, following the anchor-fixed spec (§6.1):

- the winner (softmax-sampled from the basket's candidates, each scored against `A`) always becomes the next bridge (§6.1i) — we record the full `raw_path`, unfiltered;
- alongside each winner we record `raw_sims`: its raw cosine similarity to the anchor, appended at the **same moment** as `raw_path`, so `raw_sims[i]` is the similarity for `raw_path[i + 1]`. This is the hook the filter notebooks use;
- candidates exclude both `A` and the current `bridge` (self-loop guard, §8);
- an empty-candidate basket increments `consecutive_empty` and retries from the same bridge; `max_consecutive_empty` in a row ends the walk (§6.1e);
- `visited_baskets` is per-walk, preventing repeated baskets within one walk (§8).

There is no `sentence` and no `theta` here — logging decisions belong downstream.

`run_walk` now lives in `walk_lib.py` (imported in §1) and is shared verbatim with `run_walk_cluster.py`; the mechanics above describe exactly what it does.

## 9. Generate the walks

`r` walks per starting product, sharing a single seeded RNG so the whole run is reproducible. **Every** walk is kept — including anchor-only ones with no qualifying hops — because discarding short walks is the filter notebooks' job, not this one's. Downstream reads `walk_raw_paths.jsonl` and decides what to keep.

In [ ]:
walk_rng = np.random.default_rng(random_seed)

raw_paths = []        # {"anchor_id", "raw_path", "raw_sims", "hops_taken"} — every walk
n_total = 0

# ~400k walks at this scale (20k starts x r) -> show progress over starts.
try:
    from tqdm.auto import tqdm
    start_iter = tqdm(starting_products, desc="starting products")
    _use_tqdm = True
except ImportError:
    start_iter = starting_products
    _use_tqdm = False

for i, anchor_id in enumerate(start_iter, 1):
    if not _use_tqdm and i % 500 == 0:
        print(f"  ...{i:,}/{len(starting_products):,} starting products processed")
    if anchor_id not in product_to_orders:
        continue                                    # no baskets -> cannot walk (noted in §6)
    for _ in range(r):
        w = run_walk(anchor_id, walk_rng, pid_to_idx, embeddings,
                     product_to_orders, order_to_products,
                     L, r, tau, score_transform, max_consecutive_empty)
        n_total += 1
        raw_paths.append(w)

n_with_winner = sum(1 for w in raw_paths if len(w["raw_path"]) > 1)
print(f"walks run              : {n_total:,}")
print(f"walks with >= 1 winner : {n_with_winner:,} "
      f"({100 * n_with_winner / max(n_total, 1):.1f}%)")
print(f"walks anchor-only      : {n_total - n_with_winner:,}")

## 10. Save outputs

Three artifacts. `walk_raw_paths.jsonl` is the raw corpus the filter notebooks consume; `walk_config.json` records the exact walk-mechanics settings so every run is traceable (no `theta` / `min_sentence_length` — neither applies to the walk itself); `product_basket_freq.json` hands downstream the per-product basket counts (free from `product_to_orders`) so popularity checks never have to re-read `order_products__prior.csv` or rebuild the basket graph.

In [ ]:
# 1) Raw walk paths — every walk, one line each (anchor_id, raw_path, raw_sims, hops_taken).
with open(OUT_DIR / "walk_raw_paths.jsonl", "w") as f:
    for w in raw_paths:
        f.write(json.dumps(w) + "\n")

# 2) Walk configuration — walk mechanics + resolved sample sizes + timestamp.
config = {
    "L": L,
    "r": r,
    "tau": tau,
    "score_transform": score_transform,
    "random_seed": random_seed,
    "sample_seed": sample_seed,
    "n_orders_sample": n_sample,
    "n_starting_products": len(starting_products),
    "max_consecutive_empty": max_consecutive_empty,
    "n_walks": n_total,
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
}
with open(OUT_DIR / "walk_config.json", "w") as f:
    json.dump(config, f, indent=2)

# 3) Per-product basket frequency — comes for free from product_to_orders.
#    Lets downstream answer "how common is this product in the sample" without
#    re-reading order_products__prior.csv or rebuilding the basket graph.
product_basket_freq = {
    "total_baskets": len(order_to_products),
    "product_basket_count": {int(pid): len(oids) for pid, oids in product_to_orders.items()},
}
with open(OUT_DIR / "product_basket_freq.json", "w") as f:
    json.dump(product_basket_freq, f)

print("Saved walk_raw_paths.jsonl, walk_config.json, product_basket_freq.json")
print(json.dumps(config, indent=2))
print(f"product_basket_freq: {len(product_basket_freq['product_basket_count']):,} products, "
      f"{product_basket_freq['total_baskets']:,} total baskets")

## 11. Validation / sanity checks

No filtering happens here, so the checks are about the raw walks themselves: the distribution of raw walk lengths, and a handful of full walks printed as anchor + raw path (names only — there is no "sentence" in this notebook).

In [ ]:
path_lengths = [len(w["raw_path"]) for w in raw_paths]   # anchor + winners
hop_counts = [w["hops_taken"] for w in raw_paths]         # incl. empty-candidate retries

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
if path_lengths:
    ax[0].hist(path_lengths, bins=range(1, max(path_lengths) + 2),
               align="left", color="steelblue", edgecolor="white")
ax[0].set_title("Raw path lengths (anchor + winners)")
ax[0].set_xlabel("products per raw path"); ax[0].set_ylabel("count")

if hop_counts:
    ax[1].hist(hop_counts, bins=range(0, max(hop_counts) + 2),
               align="left", color="indianred", edgecolor="white")
ax[1].set_title("Raw walk lengths (hops taken)")
ax[1].set_xlabel("hops before termination"); ax[1].set_ylabel("count")
plt.tight_layout(); plt.show()

winners_per_walk = [len(w["raw_path"]) - 1 for w in raw_paths]
print(f"mean winners logged per walk: {np.mean(winners_per_walk):.2f}")
print(f"mean hops taken per walk    : {np.mean(hop_counts):.2f}")

In [ ]:
# A few full example walks: anchor + raw path (incl. every unlogged bridge).
walks_with_winner = [w for w in raw_paths if len(w["raw_path"]) > 1]
display_rng = np.random.default_rng(0)
picks = (display_rng.choice(len(walks_with_winner),
                            size=min(10, len(walks_with_winner)), replace=False)
         if walks_with_winner else [])

for i in picks:
    w = walks_with_winner[int(i)]
    print("anchor   :", name(w["anchor_id"]))
    print("raw path :", " -> ".join(names(w["raw_path"])))
    print("hops     :", w["hops_taken"])
    print("-" * 90)